# 로직스틱 회귀

### 데이터 생성

In [10]:
import pandas as pd

df = pd.DataFrame({
    "study_hours":    [1, 2, 3, 4, 5, 2, 3, 5, 4, 1],
    "sleep_hours":    [6, 8, 5, 7, 8, 5, 8, 6, 6, 8],
    "practice_count": [10, 15, 20, 25, 30, 12, 28, 22, 32, 8],
    "score":          [47, 58, 60, 76, 87, 49, 73, 77, 79, 52],
})

X = df[["study_hours", "sleep_hours", "practice_count"]]
y = df["score"] >= 60

print(f"학생 {len(X)} 명, X : {X.shape[1]}개")
print(df[["score"]].assign(passed=y).head())

학생 10 명, X : 3개
   score  passed
0     47   False
1     58   False
2     60    True
3     76    True
4     87    True


### 데이터 분리 및 표준화

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 학습용 vs 테스트용 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X , y, test_size=0.3, random_state=42, stratify=y
)

# 표준화 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"학습 {len(X_train)}명, 테스트 {len(X_test)}명")
# iloc[행, 열]
print(f"첫 학습 학생의 공부시간: {X_train.iloc[0,0]} 시간")
print(f"=> 표준화 값: {X_train_scaled[0,0]:.2f}")


학습 7명, 테스트 3명
첫 학습 학생의 공부시간: 4 시간
=> 표준화 값: 1.21


### 모델 학습

In [17]:
from sklearn.linear_model import LogisticRegression

# 표준화된 학습데이터 + 정답
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

print("클래스:",model.classes_.tolist())
print("가중치:",model.coef_[0].round(3))

클래스: [False, True]
가중치: [ 0.873 -0.37   0.877]


### 새로운 학생데이터로 예측

In [34]:
new = pd.DataFrame([[4, 7, 30]], columns=X.columns)
new_scaled = scaler.transform(new)

predict = model.predict(new_scaled)[0]

pass_proba = model.predict_proba(new_scaled)[0, 1]

print("예측 결과:", "합격" if predict else "불합격")
print(f"합격 확률: {pass_proba:.2%}")


예측 결과: 합격
합격 확률: 92.58%


In [42]:
# 평가
from sklearn.metrics import accuracy_score, confusion_matrix

# 학습에서 사용하지 않은 학생 3명으로 예측, 평가
test_pred = model.predict(X_test_scaled)
print("실제 정답:", y_test.tolist())
print("에측 결과:", test_pred.tolist())
print(f"정확도: {accuracy_score(y_test, test_pred):.2%}")
print(confusion_matrix(y_test, test_pred, labels=[False, True]))

실제 정답: [False, True, True]
에측 결과: [False, True, True]
정확도: 100.00%
[[1 0]
 [0 2]]
